# QDoRA SFT Baseline Colab Runner

Audience:
- Someone running the disruption-classification QDoRA SFT baseline on Colab with an H100 GPU.

Prerequisites:
- The notebook is running in a directory that contains the dataset files.
- Required local files in the current working directory:
  - `sci_balanced_from2m_no_ovr.rl_balanced.jsonl`
  - `sci_balanced_from2m_no_ovr.splits.json`
- You have access to the base model you want to fine-tune from Hugging Face.

Learning goals:
- Validate the Colab runtime and dependencies.
- Materialize a local training script directly from the notebook.
- Launch the QDoRA SFT baseline with only local dataset files as inputs.
- Inspect the output artifacts before reporting issues.


## Outline

1. Confirm the runtime and install dependencies.
2. Verify that the dataset files exist in the current directory.
3. Write a local `qdora_train_local.py` file from the notebook itself.
4. Import the local trainer module and verify its defaults.
5. Configure the training run.
6. Launch training.
7. Inspect metrics and manifests.
8. Try one controlled variation before reporting issues.


In [ ]:
from __future__ import annotations

import os
import random
import subprocess
import sys
from pathlib import Path

SEED = 0
random.seed(SEED)
WORKDIR = Path.cwd()

print('python', sys.version)
print('workdir', WORKDIR)
subprocess.run(['nvidia-smi'], check=False)


## Step 1 - Install the training dependencies

Run this once after the Colab runtime starts.


In [ ]:
%pip install -q -U torch transformers peft bitsandbytes accelerate sentencepiece


## Step 2 - Verify the local dataset files

This notebook assumes the dataset assets are already present in the current working directory and does not depend on any repo layout.


In [ ]:
DATASET_PATH = WORKDIR / 'sci_balanced_from2m_no_ovr.rl_balanced.jsonl'
SPLITS_PATH = WORKDIR / 'sci_balanced_from2m_no_ovr.splits.json'

required_paths = [DATASET_PATH, SPLITS_PATH]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing required local dataset files: {missing}')

for path in required_paths:
    print(path.name, 'exists, size_mb=', round(path.stat().st_size / 1024**2, 2))


## Step 3 - Materialize or reuse the local trainer

If you already uploaded `qdora_train_local.py` into the Colab working directory, the notebook uses that file directly. Otherwise it writes the same standalone trainer into the current directory from embedded source.


In [ ]:
import base64

TRAINER_SOURCE_B64 = "..."
TRAINER_PATH = WORKDIR / 'qdora_train_local.py'

if TRAINER_PATH.exists():
    print('using uploaded trainer:', TRAINER_PATH)
else:
    TRAINER_PATH.write_bytes(bytes.fromhex(TRAINER_SOURCE_B64))
    print('wrote trainer to', TRAINER_PATH)

print('trainer_size_kb =', round(TRAINER_PATH.stat().st_size / 1024, 1))


## Step 4 - Import the local trainer

This verifies that the notebook-generated script imports cleanly and that its default paths now point at the current directory.


In [ ]:
import importlib.util
import sys

spec = importlib.util.spec_from_file_location('qdora_train_local', TRAINER_PATH)
qdora_train_local = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = qdora_train_local
spec.loader.exec_module(qdora_train_local)

print('default_dataset_path =', qdora_train_local.default_dataset_path())
print('default_splits_path  =', qdora_train_local.default_splits_path())
print('default_output_dir   =', qdora_train_local.default_output_dir())


## Step 5 - Configure the run

Build the argument namespace in Python so the later cells can call the trainer functions directly. The default notebook run is capped at 500 optimizer steps and saves checkpoints every 250 steps.


In [ ]:
import argparse

CONFIG = {
    'model_name': 'meta-llama/Llama-3.1-8B-Instruct',
    'dataset_path': DATASET_PATH,
    'splits_path': SPLITS_PATH,
    'split_source': 'seeded_jsonl',
    'train_split': 'train',
    'val_split': 'val',
    'test_split': 'test',
    'train_size': 100_000,
    'val_size': 2_000,
    'test_size': 500,
    'output_dir': WORKDIR / 'qdora_sft_runs' / 'colab_h100_500step',
    'overwrite_output_dir': True,
    'seed': 0,
    'max_length': 1024,
    'eval_max_new_tokens': 64,
    'per_device_train_batch_size': 2,
    'per_device_eval_batch_size': 8,
    'gradient_accumulation_steps': 16,
    'num_train_epochs': 1.0,
    'max_steps': 500,
    'learning_rate': 1e-5,
    'weight_decay': 0.1,
    'warmup_ratio': None,
    'warmup_steps': 15,
    'lr_scheduler_type': 'cosine',
    'max_grad_norm': 1.0,
    'logging_steps': 10,
    'save_strategy': 'steps',
    'save_steps': 250,
    'save_total_limit': 2,
    'dataloader_num_workers': 0,
    'gradient_checkpointing': True,
    'trust_remote_code': False,
    'attn_implementation': None,
    'lora_rank': 64,
    'lora_alpha': 128,
    'lora_dropout': 0.05,
    'target_modules': ','.join(qdora_train_local.DEFAULT_TARGET_MODULES),
    'teacher_confidence_mode': 'cd_index_margin',
    'report_to': 'none',
    'resume_from_checkpoint': None,
}

args = argparse.Namespace(**CONFIG)
CONFIG


## Step 6 - Prepare the dataset and tokenizer

This cell uses the trainer's split helper. By default it follows the RL-style seeded JSONL split instead of the incompatible manifest ids.


In [ ]:
qdora_train_local.set_seed(args.seed)
qdora_train_local.ensure_output_dir(args.output_dir, overwrite=args.overwrite_output_dir)

tokenizer = qdora_train_local.load_tokenizer(args)
split_to_records = qdora_train_local.load_experiment_record_splits(args)
train_records = split_to_records['train']
val_records = split_to_records['val']
test_records = split_to_records['test']

raw_train_examples = qdora_train_local.build_sft_examples(train_records, tokenizer, args.teacher_confidence_mode)
raw_val_examples = qdora_train_local.build_sft_examples(val_records, tokenizer, args.teacher_confidence_mode)
raw_test_examples = qdora_train_local.build_sft_examples(test_records, tokenizer, args.teacher_confidence_mode)

train_examples, train_filter_stats = qdora_train_local.filter_examples_by_length(tokenizer, raw_train_examples, args.max_length)
val_examples, val_filter_stats = qdora_train_local.filter_examples_by_length(tokenizer, raw_val_examples, args.max_length)
test_examples, test_filter_stats = qdora_train_local.filter_examples_by_length(tokenizer, raw_test_examples, args.max_length)

print('split_source       =', args.split_source)
print('train_records      =', len(train_records))
print('val_records        =', len(val_records))
print('test_records       =', len(test_records))
print('train_examples     =', len(train_examples))
print('val_examples       =', len(val_examples))
print('test_examples      =', len(test_examples))
print('train_filter_stats =', train_filter_stats)
print('val_filter_stats   =', val_filter_stats)
print('test_filter_stats  =', test_filter_stats)


## Step 7 - Train and evaluate directly in Python

This is the main execution cell. It loads the quantized model, trains the DoRA adapter, runs validation and test eval, and writes the manifest plus metrics.


In [ ]:
model, model_summary = qdora_train_local.load_model(args)
trainer, train_metrics = qdora_train_local.train_model(
    model=model,
    tokenizer=tokenizer,
    train_examples=train_examples,
    args=args,
    output_dir=args.output_dir,
)

trainer.model.config.use_cache = True
val_metrics = qdora_train_local.evaluate_split(
    model=trainer.model,
    tokenizer=tokenizer,
    examples=val_examples,
    split_name='val',
    args=args,
    output_dir=args.output_dir,
)
test_metrics = qdora_train_local.evaluate_split(
    model=trainer.model,
    tokenizer=tokenizer,
    examples=test_examples,
    split_name='test',
    args=args,
    output_dir=args.output_dir,
)

manifest = qdora_train_local.build_run_manifest(
    args=args,
    train_records=train_records,
    val_records=val_records,
    test_records=test_records,
    train_examples=train_examples,
    val_examples=val_examples,
    test_examples=test_examples,
    filter_stats={
        'train': train_filter_stats,
        'val': val_filter_stats,
        'test': test_filter_stats,
    },
    model_summary=model_summary,
    train_metrics=train_metrics,
    eval_metrics={'val': val_metrics, 'test': test_metrics},
)
qdora_train_local.write_json(args.output_dir / 'run_manifest.json', manifest)
qdora_train_local.write_json(args.output_dir / 'train_metrics.json', train_metrics)

print('output_dir        =', args.output_dir)
print('train_global_step =', train_metrics.get('global_step'))
print('val_label_match   =', val_metrics.get('label_match'))
print('test_label_match  =', test_metrics.get('label_match'))
print('test_macro_f1     =', test_metrics.get('macro_f1'))


In [ ]:
print('output_dir         =', args.output_dir)
print('train_examples     =', manifest['dataset']['kept_examples']['train'])
print('test_examples      =', manifest['dataset']['kept_examples']['test'])
print('train_global_step  =', train_metrics.get('global_step'))
print('latest_checkpoint  =', train_metrics.get('latest_trainer_checkpoint'))
print('test_label_match   =', test_metrics.get('label_match'))
print('test_macro_f1      =', test_metrics.get('macro_f1'))
print('test_format_strict =', test_metrics.get('format_strict_ok'))
print('test_parse_ok      =', test_metrics.get('parse_ok'))
print('test_calibration   =', test_metrics.get('calibration'))


## Pitfalls and extensions

Common failure modes:
- Hugging Face auth or model-access issues when loading the base model.
- `bitsandbytes` or CUDA mismatch in a fresh Colab runtime.
- Missing one of the two local dataset files in the current directory.
- The first full run is too large for quick debugging, so the failure signal is delayed.

Useful extensions:
- Swap the model and compare `metrics_test.json`.
- Increase `max_steps` from `500` to `1000+` once the short checkpointed run looks healthy.
- Save the traceback plus `run_manifest.json` when reporting failures back.


## Exercise

Before reporting issues, try one small controlled variation:
- cut `TRAIN_SIZE` down to `2048`
- keep `TEST_SIZE=500`
- write to a different `OUTPUT_DIR`

This helps separate environment breakage from longer-run training issues.


In [ ]:
import argparse

smoke_config = dict(CONFIG)
smoke_config['train_size'] = 2_048
smoke_config['output_dir'] = WORKDIR / 'qdora_sft_runs' / 'colab_h100_smoke_2k'
smoke_config['max_steps'] = 100
smoke_config['warmup_steps'] = 3
smoke_args = argparse.Namespace(**smoke_config)
smoke_config
